1. Load raw dataset

In [0]:
df_escorpiao = spark.table("raw_escorpiao")
display(df_escorpiao)

2. Transform data to long format (year-month)

In [0]:
from pyspark.sql.functions import col, lit
from functools import reduce

df_esc = spark.table("raw_escorpiao")

meses_validos = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

mapa_anos = {
    2020: ("_c2", "_c3"),
    2021: ("_c5", "_c6"),
    2022: ("_c8", "_c9"),
    2023: ("_c11", "_c12"),
    2024: ("_c14", "_c15"),
    2025: ("_c17", "_c18"),
    2026: ("_c20", "_c21"),
}

dfs = []

for ano, (col_casos, col_obitos) in mapa_anos.items():
    df_ano = (
        df_esc
        .filter(col("_c0").isin(meses_validos))
        .select(
            col("_c0").alias("mes"),
            lit(ano).alias("ano"),
            col(col_casos).cast("int").alias("casos"),
            col(col_obitos).cast("int").alias("obitos")
        )
    )
    dfs.append(df_ano)

df_long = reduce(lambda a, b: a.unionByName(b), dfs)

display(df_long)

3. Create chronological month order

In [0]:
from pyspark.sql.functions import when

df_long_ord = df_long.withColumn(
    "ordem_mes",
    when(col("mes") == "JANEIRO", 1)
    .when(col("mes") == "FEVEREIRO", 2)
    .when(col("mes") == "MARÇO", 3)
    .when(col("mes") == "ABRIL", 4)
    .when(col("mes") == "MAIO", 5)
    .when(col("mes") == "JUNHO", 6)
    .when(col("mes") == "JULHO", 7)
    .when(col("mes") == "AGOSTO", 8)
    .when(col("mes") == "SETEMBRO", 9)
    .when(col("mes") == "OUTUBRO", 10)
    .when(col("mes") == "NOVEMBRO", 11)
    .when(col("mes") == "DEZEMBRO", 12)
)

display(df_long_ord.orderBy("ano", "ordem_mes"))

4. Replace nulls with zeros

In [0]:
from pyspark.sql.functions import when, col

df_long_ord = df_long_ord.withColumn(
    "obitos",
    when((col("ano") == 2024) & (col("obitos").isNull()), 0)
    .otherwise(col("obitos"))
)
display(df_long_ord.orderBy("ano", "ordem_mes"))

5. Validate final schema

In [0]:
df_long_ord.printSchema()

6. Validate aggregated data by year

In [0]:
display(
    df_long_ord.groupBy("ano").sum("casos", "obitos")
)

7. Save transformed table

In [0]:
df_long_ord.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.fato_escorpiao")